> **Requirements:** Python 3 with `numpy` and `matplotlib`.
>
> **Outputs:** figures are written to `./figures/` and data tables to `./data/`, created next to this notebook.

# A techno-enviro-economic assessment of decarbonising UK industrial bread baking

*Integrated techno-economic and life-cycle model of a representative UK continuous tunnel oven gas, gas-plus-efficiency, electric-resistance and hybrid electric-resistance-infrared configurations, with Monte-Carlo uncertainty across four NESO grid pathways.*

MSc Renewable Energy and Clean Technology, The University of Manchester (2026). Student ID 11551506.

### Abstract
The UK industrial baking sector produces about 2.5 Mt of baked goods a year, consuming 2,000 GWh of energy and emitting 570,000 tCO₂e. Baking energy consumption is mostly for process heat, which is low to medium-temperature and highly electrifiable. UK industrial electricity costs about 4.84 times more than gas, limiting cost competitiveness, while the reducing grid carbon intensity favours electrification. This study develops an integrated, plant-level techno-economic and life-cycle model of a representative UK continuous tunnel oven for gas, gas-plus-efficiency, electric-resistance and electric-resistance plus infrared configurations, across four National Energy System Operator grid pathways. Electrification was determined as the lower-carbon option in every pathway, cutting lifetime emissions from 13.9 ktCO₂e for gas to 1.9–5.75 ktCO₂e (59–86%). Electric ovens are marginally cleaner at the 2024 grid on an average basis, but this margin reverses on a marginal basis. Over 15 years, electrification is decidedly the low carbon option. Economically, the electric oven's net present cost was about 2.3 times that of gas, and for parity, the ratio of electricity to gas price needs to fall from 4.84 to about 1.18. Alternatively, a carbon price near £564/tCO₂e, about fifteen times the current traded value is required. Efficiency-first measures were shown as the least-cost and lowest-emitting configuration at today's grid, making it the no-regret near-term move.  Demand-side flexibility was found to save about 7.6% of annual electricity cost and about 0.5% of emissions on the marginal basis, which implied flexibility may be pursued as a cost saving measure in electric ovens, rather than for emissions reduction. The electricity-to-gas price ratio was the most important factor limiting electrification, while oven capital cost is not a binding constraint. By reporting break-even thresholds at plant level rather than sector-wide estimates, this study gives food manufacturers, equipment suppliers and policymakers a transparent, transferable basis for deciding when and where to electrify.

### Objectives
1.	To determine the energy demand, cost, and direct emissions of a representative gas-fired baking oven as a baseline, including the effect of efficiency-first measures.
2.	To quantify the capital and operating costs of electrified and hybrid oven configurations under a range of electricity-to-gas price ratios and tariff structures.
3.	To quantify the life-cycle greenhouse-gas emissions of each configuration under present and projected grid carbon-intensity scenarios.
4.	To determine the additional economic and emissions impact of demand-side flexibility on the oven operation.
5.	To determine the conditions such as price ratio, carbon price, and grid carbon intensity at which electrification becomes the least-cost or least-carbon option based on 2 – 4.




In [ ]:
import os
if "__file__" not in globals():
    __file__ = os.path.join(os.getcwd(), "tea_lca_model.py")


## 1&nbsp;&nbsp;Setup, imports and global plotting configuration

In [ ]:
# -*- coding: utf-8 -*-


import json
import csv
import os
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch

warnings.filterwarnings("ignore",message="All-Nan slice encountered")


HERE = os.path.dirname(os.path.abspath(__file__))
FIG_DIR = os.path.join(HERE, "figures")
CSV_DIR = os.path.join(HERE, "data")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

#Plot Global Paramters

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "figure.dpi": 900,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "xtick.top": True,
    "ytick.right": True,
})

GBP = "GBP"

COLOUR = {
    "gas":     "#8c6d3f",
    "gas_eff": "#c2a878",
    "elec":    "#2f6f9f",
    "hybrid":  "#3a9d6b",
    "carbon":  "#b0483b",
}

#Monte Carlo Draws
N_MC = 10000 #Draws for scalar output
N_MC_CURVE = 10000 #Draws for curve bands



#Oven configurations

GAS     = "Gas (baseline)"
GAS_EFF = "Gas + efficiency"
ELEC    = "Electric (ER)"
HYBRID  = "Hybrid (ER+IR)"


CONFIGS = [GAS, GAS_EFF, ELEC, HYBRID]
FUEL = {GAS: "gas", GAS_EFF: "gas", ELEC: "elec", HYBRID: "elec"}

LIFE      = 15        # years under consideraation
HOURS     = 4992      # operating h/yr (16 h/day x 6 d/wk x 52)
REF_YEAR  = 2024
YEARS     = np.arange(REF_YEAR + 1, REF_YEAR + 1 + LIFE)   # 2025..2039
IR_WEIGHT = 0.50      # hybrid SEC = 50% ER + 50% IR

#Price and Carbon intensity

# Real price/carbon trajectories are built PER draw from sampled growth rates
# (growth_path below): year 1 = base year, then compounds in REAL terms (1a).
def growth_path(g):
    return (1.0 + g) ** np.arange(LIFE)   # [1, (1+g), (1+g)^2, ... ] length LIFE

#Climate Change Levy

CCL_RATES = {"gas": 0.00775, "elec": 0.00775}    # CCL rates per GBP/kWh
PRICES_INCLUDE_CCL = False   # p_gas and p_elec are exlusve of CCL which is added explicitly
CCL_PATH = np.ones(LIFE)

#CCL with Climate Change Agreement Discounts
CCA_SHARE_PAID = {"gas": 0.11, "elec": 0.08}



#MODEL INPUT RANGES (LOW, BASE, HIGH)

RANGE = {
    "production":      (0.75*3.2*4992e3, 3.2*4992e3, 1.25*3.2*4992e3),  # in kg/yr; base 3.2 t/h x 4,992 h, calculated from Carbon Trust 2016).  Range: assumed +/-25%
    "price_gas":       (0.030,  0.037,  0.060),   # GBP/kWh (DESNZ 2024)
    "spark_gap":       (1.5,    4.84,   6.5),     # sampled DIRECTLY basing on p_elec = SG x p_gas, Base 4.84 = 0.179/0.037.
    "ef_gas_direct":   (0.180,  0.18290, 0.186),  # Scope 1 EF
    "ef_gas_lc":       (0.200,  0.21311, 0.230),  # gas life-cycle (combustion+WTT) EF
    "ef_grid_2024":    (0.171,  0.22535, 0.300),  # 2024 grid EF
    "eff_saving":      (0.05,   0.15,   0.25),    # efficiency-first saving e, #Low based on (Carbon Trust 2016: 4.7%; Paton 2013), #high (Therkelsen 2014; Spooner 2026 heat-recovery case 7.5%).
    "elec_saving":     (0.05,   0.13,   0.20),    # electric-vs-gas SEC saving d: SEC_elec = SEC_gas*(1-d).
    # Base from Roosen (1993): 5.34 (electric) vs 6.17 (gas) MJ/kg, via Ladha-Sabur et al.(2019) 
    # Range spans the 10-20% manufacturer case
    #   studies in Tannous et al. (2026).  Sampled as a RATIO so electric never exceeds gas.
    "sec_ir":          (0.24,   0.265,  0.29),    # IR component of hybrid SEC (kWh/kg) approx. 12% above the electric SEC base
    "om_fraction":     (0.02,   0.03,   0.05),    # fixed O&M fraction of oven CAPEX
    "discount_rate":   (0.035,  0.08,   0.15),    # real; 3.5% from Green Book to 15% highested with 8% base
    "carbon_price":    (28.0,   37.0,  46.0),   # Traded Carbon carbon price GBP/tCO2e, Scope 1 only, policy scenario, # base = DESNZ (2025) UK ETS civil-penalty price; high = DESNZ (2026) traded value.
    # 1a: dynamic REAL trajectories - growth rates sampled, so uncertainty flows into the MC/SRC
    "carbon_growth":   (0.00,   0.05,   0.09),   # real %/yr rise in carbon price (DESNZ traded carbon values 2024)
    "gas_growth":      (-0.01,  0.00,   0.02),   # real %/yr gas-price drift; 0 = flat real (default)
    "elec_growth":     (-0.02,  0.00,   0.02),   # real %/yr elec-price drift; differing gas/elec => spark gap drifts over life
    "capacity_charge": (14.0,   15.3,   20.0),    # q, GBP/kW/yr from DUoS/TNUoS schedule
    "grid_connection": (0.3e6,  0.65e6, 1.0e6),   # site-specific CAPEX, electric options
    "residual_fraction": (0.0,  0.05,   0.10),    # residual asset value at year 15, fraction of oven CAPEX
    # Demand-side flexibility
    "shiftable":       (0.05,   0.25,   0.50),    # deferrable fraction s; near-zero low per continuous-line evidence
    "price_factor":    (0.56,   0.68,   0.80),    # k_p; to be replaced by half-hourly price distributions
    "emis_factor":     (0.63,   0.71,   0.82),    # k_e, AVERAGE-EF basis; from half-hourly NESO data
    "ef_marginal":     (0.35,   0.45,   0.55),    # marginal grid EF (gas-fired margin), kg/kWh
    "emis_factor_marg": (0.92,  0.98,   1.00),    # k_e on the MARGINAL basis (margin often gas in both periods)
    "peak_factor":     (0.85,   0.95,   1.00),    # k_D: peak-demand (capacity-charge) factor under flexible operation
}


## 2&nbsp;&nbsp;Oven specific energy consumption (SEC) and capital cost (CAPEX) inputs
SEC and CAPEX ranges for gas, gas-plus-efficiency, electric-resistance and hybrid (ER+IR) ovens; sampled as triangular distributions in the Monte Carlo.

In [ ]:
# OVEN SPECIFIC ENERGY CONSUMPTION AND CAPEX

#SEC

SEC_RANGE = {
    GAS: (0.221, 0.272, 0.317),
}

#CAPEX

CAPEX_RANGE = {
    GAS:     (0.9*0.90e6,  0.90e6,  1.1*0.90e6),
    GAS_EFF: (0.9*1.035e6,  1.035e6, 1.1*1.035e6),
    ELEC:    (0.9*0.9936e6,  0.9936e6,  1.1*0.9936e6),
    HYBRID:  (0.9*0.9936e6,  0.9936e6,  1.1*0.9936e6),
}

#GRID INTENSITY SCENARIOS

# Grid EF for emissions-figure annotations & the counterfactual line presentation only. Not used for MC grid input).

GRID_SCENARIOS = {
    "UK 2024 base (DEFRA)":          0.22535,
    "UK operational 2023 (NESO)*":   0.133,   # *operational basis
    "UK 2030 (NESO)":                0.073,
    "UK 2035 (NESO, floored at 0)":  0.0,     # NESO -0.011 floored: BECCS accounting
    "High-carbon (counterfactual)":  0.134,   # No further grid decarbonisation
}



## 3&nbsp;&nbsp;Grid carbon-intensity pathways (NESO FES 2024)
Four NESO Future Energy Scenarios with 2025/2030/2035/2039 anchor points, linearly interpolated to yearly grid emission factors. Each pathway is carried through the full Monte Carlo as a scenario axis (deep uncertainty), not sampled probabilistically.

In [ ]:
# NESO FES 2024 Grid intensity scenarions with 2025, 2030, 2035, and 2039 anchor ponts, linearly interpolated for yearly values.

# to the numbers - it just carries whatever four paths are defined here.
_SCEN_ANCHORS = {
    "Holistic Transition":             {2023: 0.133, 2030: 0.041, 2035: -0.017, 2050: -0.028},
    "Electric Engagement":             {2023: 0.133, 2030: 0.073, 2035: -0.011, 2050: -0.036},
    "Hydrogen Evolution":              {2023: 0.133, 2030: 0.074, 2035: -0.009, 2050: -0.036},
    "Counterfactual (Falling Short)":  {2023: 0.133, 2030: 0.134, 2035: 0.069, 2050: 0.021},
}
def _interp_traj(anchors):
    ys = np.array(sorted(anchors), dtype=float)
    vs = np.array([anchors[int(y)] for y in ys], dtype=float)
    return np.clip(np.interp(YEARS.astype(float), ys, vs), 0.0, None)
GRID_SCENARIOS_TRAJ = {name: _interp_traj(a) for name, a in _SCEN_ANCHORS.items()}

# Grid intensity trajectory Chosen as the cental scenario

CENTRAL_SCENARIO = "Electric Engagement"
BASE_GRID_TRAJ = GRID_SCENARIOS_TRAJ[CENTRAL_SCENARIO]
GRID_EF_PROJECTION = {int(YEARS[k]): float(BASE_GRID_TRAJ[k]) for k in range(len(YEARS))}

#Parity Comparision between reference case and Counterfactual
GRID_EF_2024_REF = RANGE["ef_grid_2024"][1]   # 2024 grid-EF reference
SLOW_GRID_TRAJ = np.full(LIFE, GRID_SCENARIOS["High-carbon (counterfactual)"])  # flat high-carbon counterfactual


#Bulding Paramter set (P)

def _assemble(P, sec_gas):

    P["price_elec"] = P["spark_gap"] * P["price_gas"]
    P.setdefault("ccl_gas",  CCL_RATES["gas"])    # statutory fixed, not sampled (A)
    P.setdefault("ccl_elec", CCL_RATES["elec"])
    sec_elec = sec_gas * (1 - P["elec_saving"])
    P["sec"] = {
        GAS:     sec_gas,
        GAS_EFF: sec_gas * (1 - P["eff_saving"]),
        ELEC:    sec_elec,
        HYBRID:  (1 - IR_WEIGHT) * sec_elec + IR_WEIGHT * P["sec_ir"],
    }
    return P


def base_params():
    P = {key: rng[1] for key, rng in RANGE.items()}
    P["capex"] = {c: CAPEX_RANGE[c][1] for c in CONFIGS}
    return _assemble(P, SEC_RANGE[GAS][1])


def sample_params(rng):
    
    P = {key: rng.triangular(lo, mid, hi) for key, (lo, mid, hi) in RANGE.items()}
    P["capex"] = {c: rng.triangular(*CAPEX_RANGE[c]) for c in CONFIGS}
    sec_gas = rng.triangular(*SEC_RANGE[GAS])
    P["sec_gas_raw"] = sec_gas
    return _assemble(P, sec_gas)


def override(P, **changes):
    Q = dict(P)
    Q.update(changes)
    if "price_elec" not in changes and ("spark_gap" in changes or "price_gas" in changes):
        Q["price_elec"] = Q["spark_gap"] * Q["price_gas"]
    return Q


BASE_P = base_params()
rng = np.random.default_rng(42)
MC_PARAMS = [sample_params(rng) for _ in range(N_MC)]
MC_CURVE  = MC_PARAMS[:N_MC_CURVE]


## 4&nbsp;&nbsp;Model formulation
Annual energy, direct and life-cycle emissions, operating cost, net present cost (NPC), break-even spark gap and carbon price, MAC and LCOH.

In [ ]:
# CORE FORMULA


def is_gas(config):
    return FUEL[config] == "gas"


def discount_factors(P):
    return 1.0 / (1.0 + P["discount_rate"]) ** np.arange(1, LIFE + 1)

#A = [1-(1+r)^-N]/r (eq 3.7), i.e. the sum of the discount factors
def annuity_factor(P):

    return discount_factors(P).sum()

#E = SEC x P (kWh/yr)
def annual_energy(P, config):

    return P["sec"][config] * P["production"]

#Peak electrical demand D (kW): continuous line, so D = E/hours
def peak_demand(P, config):

    return annual_energy(P, config) / HOURS if not is_gas(config) else 0.0

#Scope 1 emissions (tCO2e/yr): gas combustion only
def annual_emissions_direct(P, config):
    ef = P["ef_gas_direct"] if is_gas(config) else 0.0
    return annual_energy(P, config) * ef / 1000.0

#Year-by-year grid EF: the 15-year INPUT projection
def grid_trajectory(P):
    return BASE_GRID_TRAJ * (P["ef_grid_2024"] / GRID_EF_2024_REF)

#Operational + upstream emissions (tCO2e/yr,) at a single grid EF
def annual_emissions_lc(P, config, ef_grid=None):
    ef = P["ef_gas_lc"] if is_gas(config) else (P["ef_grid_2024"] if ef_grid is None else ef_grid)
    return annual_energy(P, config) * ef / 1000.0

#Lifetime operational + upstream emissions (tCO2e over 15 yr)
def lifetime_emissions(P, config, traj=None):
    E = annual_energy(P, config)
    if is_gas(config):
        return E * P["ef_gas_lc"] * LIFE / 1000.0
    traj = grid_trajectory(P) if traj is None else traj
    return E * traj.sum() / 1000.0

#CCL charged on both gas and electric consumption
def ccl_rate(P, config):
    return P["ccl_gas"] if is_gas(config) else P["ccl_elec"]

#Annual OPEX
def opex_components(P, config, carbon_price=None):
    cp = P["carbon_price"] if carbon_price is None else carbon_price
    E = annual_energy(P, config)
    if is_gas(config):
        energy, capacity = E * P["price_gas"], 0.0
    else:
        p_energy = max(P["price_elec"] - P["capacity_charge"] / HOURS, 0.0)
        energy   = E * p_energy
        capacity = P["capacity_charge"] * peak_demand(P, config)
    levy   = E * ccl_rate(P, config)                      # CCL, both carriers (A)
    carbon = annual_emissions_direct(P, config) * cp      # Scope 1 only (B)
    om     = P["om_fraction"] * P["capex"][config]
    return energy, capacity, levy, carbon, om


def annual_opex(P, config, carbon_price=None):
    return sum(opex_components(P, config, carbon_price))

#Year-0 outlay: oven CAPEX plus grid connection exl.  Any downtime for changeover is ignored
def capex_initial(P, config):
    extra = 0.0 if is_gas(config) else P["grid_connection"]
    return P["capex"][config] + extra

#Net present cost of ownership (GBP), NPC = CAPEX0 + sum_t OPEX_t/(1+r)^t - residual value, t = 1..15.OPEX_t is built from year-indexed price/carbon paths (flat real by default)


## 5&nbsp;&nbsp;Carbon-price treatment
Either the hard-coded DESNZ traded-carbon trajectory (`path`) or a constant real growth rate `g` (`growth`). Real price/carbon trajectories are built per Monte-Carlo draw so their uncertainty flows into the results.

In [ ]:
CARBON_MODE = "path"                # "path" = DESNZ year-by-year (below) | "growth" = constant g (carbon_growth)
CENTRAL_CARBON_SERIES = "net_zero"  # central path in "path" mode: "net_zero" (policy-aligned) or "market" (near-term ETS)

# DESNZ traded carbon values 2025-2039, real 2024 GBP/tCO2e (Carbon Price Growth.xlsx)
CARBON_SERIES = {
    "market":   np.array([44, 62, 75, 88, 80, 78, 85, 91, 97, 100, 109, 115, 122, 130, 132], float),
    "low":      np.array([49, 62, 66, 62, 53, 50, 54, 60, 63,  65,  72,  77,  84,  94,  96], float),
    "net_zero": np.array([63, 87, 87, 88, 80, 78, 85, 91, 97, 100, 109, 115, 122, 130, 132], float),
    "high":     np.array([74, 103, 104, 110, 105, 107, 114, 118, 123, 125, 134, 140, 146, 153, 153], float),
}

def carbon_price_path(P):
    # per-draw DESNZ path (GBP/t, 15 yr); sampled carbon_price positions between low/central/high
    lo, _mid, hi = RANGE["carbon_price"]
    frac = min(max((P["carbon_price"] - lo) / (hi - lo), 0.0), 1.0)   # 0 at 28, 0.5 at 37, 1 at 46
    central = CARBON_SERIES[CENTRAL_CARBON_SERIES]
    if frac <= 0.5:
        return CARBON_SERIES["low"] + (frac / 0.5) * (central - CARBON_SERIES["low"])
    return central + ((frac - 0.5) / 0.5) * (CARBON_SERIES["high"] - central)

def carbon_cost_vector(P, config, cp, cp_is_override):
    # 15-yr carbon COST (GBP/yr), gas only.  An explicit carbon_price (override) forces the
    # constant-price growth basis, so break-even / decision-map analyses keep their meaning.
    if not is_gas(config) or cp == 0.0:
        return np.zeros(LIFE)
    g_direct = annual_emissions_direct(P, config)                    # tCO2e/yr
    if CARBON_MODE == "path" and not cp_is_override:
        return g_direct * carbon_price_path(P)                       # DESNZ trajectory
    return g_direct * cp * growth_path(P["carbon_growth"])           # constant real growth g

def npc_total(P, config, carbon_price=None):
    df = discount_factors(P)
    cp = P["carbon_price"] if carbon_price is None else carbon_price
    energy, capacity, levy, _carbon, om = opex_components(P, config, cp)
    price_path = growth_path(P[FUEL[config] + "_growth"])            # gas_growth / elec_growth
    carbon_t   = carbon_cost_vector(P, config, cp, carbon_price is not None)
    opex_t = energy * price_path + capacity + levy * CCL_PATH + carbon_t + om
    residual = P["residual_fraction"] * P["capex"][config] * df[-1]
    return capex_initial(P, config) + (opex_t * df).sum() - residual

#NPC(electric) - NPC(gas) in GBP million (negative favours electric)
def npc_diff_elec_gas(P, carbon_price=None):

    return (npc_total(P, ELEC, carbon_price) - npc_total(P, GAS, carbon_price)) / 1e6


#Spark gap at which electric energy+levy+carbon cost equals gas (eq 3.13,

def breakeven_spark_gap(P, carbon_price, gas_config=GAS, elec_config=ELEC):
    E_gas, E_elec = annual_energy(P, gas_config), annual_energy(P, elec_config)
    gas_cost = (E_gas * (P["price_gas"] + P["ccl_gas"])
                + annual_emissions_direct(P, gas_config) * carbon_price)
    return (gas_cost - E_elec * P["ccl_elec"]) / (E_elec * P["price_gas"])

#Spark gap at which NPC(electric) = NPC(gas) - full lifetime basis, for

def breakeven_spark_gap_npc(P, carbon_price=None):
    f = lambda s: npc_diff_elec_gas(override(P, spark_gap=s), carbon_price)
    y1, y2 = f(1.0), f(5.0)
    return 1.0 + (5.0 - 1.0) * (0.0 - y1) / (y2 - y1) if y2 != y1 else np.nan

#Constant carbon price c* equalising the two NPCs. Carbon is charged on Scope 1 only
def breakeven_carbon_price(P):
    d_npc0 = npc_total(P, ELEC, carbon_price=0.0) - npc_total(P, GAS, carbon_price=0.0)
    pv_direct = annual_emissions_direct(P, GAS) * (discount_factors(P) * growth_path(P["carbon_growth"])).sum()
    return d_npc0 / pv_direct if pv_direct > 0 else np.nan

#MAC (GBP/tCO2e) at a static grid EF: annualised extra cost of electrifying divided by annual operational+upstream emissions saved. Carbon price excluded from the cost side; denominator is the true abatement

def mac_value(P, ef_grid):

    d_cost = (npc_total(P, ELEC, 0.0) - npc_total(P, GAS, 0.0)) / annuity_factor(P)
    d_emis = annual_emissions_lc(P, GAS) - annual_emissions_lc(P, ELEC, ef_grid=ef_grid)
    return d_cost / d_emis if d_emis > 0 else np.nan

#Lifetime-trajectory MAC: delta NPC (no carbon) over lifetime emissions saved along the NESO trajectory - the decision-relevant abatement cost on a decarbonising grid

def mac_lifetime(P, traj=None):
    d_npc = npc_total(P, ELEC, 0.0) - npc_total(P, GAS, 0.0)
    d_emis = lifetime_emissions(P, GAS) - lifetime_emissions(P, ELEC, traj=traj)
    return d_npc / d_emis if d_emis > 0 else np.nan

#Levelised cost of heat/output: NPC over discounted output.Denominator convention: per kg product (primary) or per kWh final energy. (Per unit useful heat requires heat-utilisation data: future work.)

def lcoh(P, config, per="kg", include_carbon=True):

    cp = P["carbon_price"] if include_carbon else 0.0
    out = P["production"] if per == "kg" else annual_energy(P, config)
    return npc_total(P, config, cp) / (out * annuity_factor(P))

#EF* (kg/kWh): grid EF below which electric beats gas on emissions.
def crossover_grid_ef(P): 
    return P["sec"][GAS] * P["ef_gas_lc"] / P["sec"][ELEC]

#DEMAND SIDE FLEXIBILITY

#Annual electricity cost under flexible operation
    
def flex_cost(P, config):
    E = annual_energy(P, config)
    p_energy = max(P["price_elec"] - P["capacity_charge"] / HOURS, 0.0)
    blend = (1 - P["shiftable"]) + P["shiftable"] * P["price_factor"]
    return (E * p_energy * blend + P["capacity_charge"] * peak_demand(P, config) * P["peak_factor"]+ E * ccl_rate(P, config))

#Annual electricity cost, firm operation (= p_elec x E + CCL x E)
def firm_cost(P, config):
    energy, capacity, levy, _, _ = opex_components(P, config, 0.0)
    return energy + capacity + levy

#Annual emissions under flexible operation 
def flex_emis(P, config, basis="average"):
    E = annual_energy(P, config)
    if basis == "average":
        ef, ke = P["ef_grid_2024"], P["emis_factor"]
    else:
        ef, ke = P["ef_marginal"], P["emis_factor_marg"]
    return E * ef * ((1 - P["shiftable"]) + P["shiftable"] * ke) / 1000.0


def firm_emis(P, config, basis="average"):
    ef = P["ef_grid_2024"] if basis == "average" else P["ef_marginal"]
    return annual_energy(P, config) * ef / 1000.0


#Uncertainty terms

#Base value plus P5/P50/P95 across the Monte Carlo draws.
def scalar_band(eval_fn, params=MC_PARAMS):
    base = eval_fn(BASE_P)
    draws = np.array([eval_fn(P) for P in params])
    p5, p50, p95 = np.nanpercentile(draws, [5, 50, 95])
    return base, p5, p50, p95


def curve_band(x_values, eval_fn, params=MC_CURVE):
    base = np.array([eval_fn(BASE_P, x) for x in x_values])
    draws = np.array([[eval_fn(P, x) for x in x_values] for P in params])
    p5, p95 = np.nanpercentile(draws, 5, axis=0), np.nanpercentile(draws, 95, axis=0)
    return base, p5, p95

#Asymmetric [low, high] error bars from (base, p5, p50, p95)
def err_bars(bands):
    low  = [max(0.0, b[0] - b[1]) for b in bands]
    high = [max(0.0, b[3] - b[0]) for b in bands]
    return [low, high]


def wrap_labels(configs):
    return [c.replace(" (", "\n(") for c in configs]


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, filename), bbox_inches="tight")
    plt.close(fig)


def write_csv(filename, header, rows):
    with open(os.path.join(CSV_DIR, filename), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

#Representative (low, base, high) SEC per config

def sec_display(config):
    g = SEC_RANGE[GAS]
    if config == GAS:
        return g
    if config == GAS_EFF:
        e = RANGE["eff_saving"]
        return (g[0] * (1 - e[2]), g[1] * (1 - e[1]), g[2] * (1 - e[0]))
    d = RANGE["elec_saving"]
    elec = (g[0] * (1 - d[2]), g[1] * (1 - d[1]), g[2] * (1 - d[0]))
    if config == ELEC:
        return elec
    ir = RANGE["sec_ir"]
    return tuple((1 - IR_WEIGHT) * elec[i] + IR_WEIGHT * ir[i] for i in range(3))


results = {
    "currency": "real 2024 GBP",
    "life_years": LIFE,
    "method": "dynamic NPC (year-by-year, 2025-2039); triangular MC, spark gap sampled directly",
    "carbon_pricing": ("Scope 1 (on-site combustion) only, policy scenario / shadow price; "
                       "NOT applied to electricity - UK ETS costs are embedded in the retail "
                       "tariff via generator pass-through"),
    "levies_CCL": {"treatment": ("CCL charged on BOTH gas and electricity consumption at "
                                 "2024-25 main rates (the instrument the plant actually pays, "
                                 "methods 2.7); prices are ex-CCL; statutory rates, not sampled"),
                   "rate_gas_GBP_per_kWh": CCL_RATES["gas"],
                   "rate_elec_GBP_per_kWh": CCL_RATES["elec"],
                   "prices_include_ccl": PRICES_INCLUDE_CCL},
    "base_spark_gap": RANGE["spark_gap"][1],
    "base_price_ratio_incl_CCL": ((RANGE["spark_gap"][1] * RANGE["price_gas"][1] + CCL_RATES["elec"])
                                  / (RANGE["price_gas"][1] + CCL_RATES["gas"])),
    "n_monte_carlo": N_MC,
    "grid_EF_projection_input": {
        "basis": "kg CO2e/kWh, DEFRA-harmonised (Table 3.3); 15-year model INPUT (v5), "
                 "default = NESO-derived trajectory; MC rescales it via ef_grid_2024",
        "values": dict(zip(YEARS.astype(str), np.round(BASE_GRID_TRAJ, 4))),
    },
    "ranges_scalar": RANGE,
    "ranges_SEC": {c: sec_display(c) for c in CONFIGS},
    "ranges_CAPEX_oven": CAPEX_RANGE,
    "energy": {}, "emissions": {}, "cost_base": {}, "npc": {},
}




## 6&nbsp;&nbsp;Verification and validation
Internal consistency checks (SEC ratios, price identities, NPC linearity in carbon price) run as assertions before any results are produced.

In [ ]:
# VERIFICATION AND VALIDATION (3.10)


print("VERIFICATION & VALIDATION")

# Unit / consistency checks
assert abs(BASE_P["price_elec"] - 0.179) < 1e-3, "base p_elec != Table 3.2"
assert abs(BASE_P["sec"][GAS_EFF] - 0.272 * 0.85) < 1e-9, "SEC ratio broken"
assert abs(BASE_P["sec"][ELEC] - 0.272 * 0.87) < 1e-9, "electric SEC ratio broken"
assert abs(BASE_P["sec"][HYBRID] - (0.5 * BASE_P["sec"][ELEC] + 0.5 * BASE_P["sec_ir"])) < 1e-9, "hybrid mix broken"
assert all(P["sec"][ELEC] < P["sec"][GAS] for P in MC_PARAMS), "electric SEC must stay below gas"
e_, cap_, lev_, carb_, om_ = opex_components(BASE_P, ELEC, 0.0)
assert abs((e_ + cap_) - BASE_P["price_elec"] * annual_energy(BASE_P, ELEC)) < 1.0, \
    "electricity price decomposition does not close"
# NPC linearity in carbon price (used by decision-map isolines)
n0, n1 = npc_total(BASE_P, GAS, 0.0), npc_total(BASE_P, GAS, 100.0)
pv_dir = annual_emissions_direct(BASE_P, GAS) * (discount_factors(BASE_P) * growth_path(BASE_P["carbon_growth"])).sum()
assert abs((n1 - n0) - 100.0 * pv_dir) < 1.0, "NPC not linear in carbon price"
print("Unit and energy-balance closure checks: PASS")
_cser = CARBON_SERIES[CENTRAL_CARBON_SERIES]
print(f"[Carbon] mode={CARBON_MODE}; central series={CENTRAL_CARBON_SERIES} "
      f"({_cser[0]:.0f} in 2025 -> {_cser[-1]:.0f} in 2039 GBP/t); "
      f"growth-mode g base={RANGE['carbon_growth'][1]*100:.0f}%/yr")

# Grid-projection input checks
assert sorted(GRID_EF_PROJECTION) == [int(y) for y in YEARS], \
    "GRID_EF_PROJECTION must hold exactly the operating years 2025-2039"
assert np.all(np.isfinite(BASE_GRID_TRAJ)) and np.all(BASE_GRID_TRAJ >= 0.0), \
    "grid projection values must be finite and non-negative"
assert abs(lifetime_emissions(BASE_P, ELEC)
           - annual_energy(BASE_P, ELEC) * BASE_GRID_TRAJ.sum() / 1000.0) < 1e-6, \
    "lifetime emissions must integrate the input projection year by year"
print(f"Grid-projection input (v5): 15 yearly values 2025-2039, "
      f"mean {BASE_GRID_TRAJ.mean():.4f} kg/kWh, 2025 {BASE_GRID_TRAJ[0]:.4f} -> "
      f"2039 {BASE_GRID_TRAJ[-1]:.4f}: PASS")

# Policy-cost checks
# (i) ETS carbon price must NOT touch electricity: the carbon OPEX component of
#     the electric and hybrid ovens is zero at ANY carbon price.
for cfg in (ELEC, HYBRID):
    comp136 = opex_components(BASE_P, cfg, 136.0)
    assert comp136[3] == 0.0, f"ETS carbon price leaked into electricity OPEX ({cfg})"
    assert abs(npc_total(BASE_P, cfg, 136.0) - npc_total(BASE_P, cfg, 0.0)) < 1e-6, \
        f"NPC of {cfg} must be invariant to the carbon price"
# (ii) CCL closure: levy component equals CCL rate x annual energy, both carriers.
for cfg in CONFIGS:
    lev = opex_components(BASE_P, cfg, 0.0)[2]
    assert abs(lev - ccl_rate(BASE_P, cfg) * annual_energy(BASE_P, cfg)) < 1e-6, \
        f"CCL closure failed ({cfg})"
assert opex_components(BASE_P, GAS, 0.0)[2] > 0 and opex_components(BASE_P, ELEC, 0.0)[2] > 0, \
    "CCL must be charged on BOTH carriers (methods 2.7)"
# (iii) Break-even spark gap consistency: at SG = SG*(c), gas and electric
#       energy+levy+carbon annual costs must be equal.
_sg_star = breakeven_spark_gap(BASE_P, BASE_P["carbon_price"])
_Q = override(BASE_P, spark_gap=_sg_star)
_g = sum(opex_components(_Q, GAS)[0:4])    # energy + capacity + levy + carbon
_e = sum(opex_components(_Q, ELEC)[0:4])
assert abs(_g - _e) < 1.0, "break-even spark gap does not equalise energy+levy+carbon costs"
_ratio_incl = (BASE_P["price_elec"] + BASE_P["ccl_elec"]) / (BASE_P["price_gas"] + BASE_P["ccl_gas"])
print(f"Policy-cost checks (2.7): carbon on Scope 1 only (zero for electric at c=136): PASS; "
      f"CCL on both carriers at {CCL_RATES['gas']*100:.3f} p/kWh: PASS; "
      f"SG* self-consistency: PASS")
print(f"Ex-CCL spark gap {BASE_P['spark_gap']:.2f} -> CCL-inclusive price ratio {_ratio_incl:.2f} "
      f"(CCL narrows the effective gap because the equal per-kWh levy weighs more on cheap gas)")

# Benchmarks against published values (validation)
sec_gas = BASE_P["sec"][GAS]
per_kg_emis = sec_gas * BASE_P["ef_gas_lc"]          # kg CO2e/kg baked
per_kg_cost = sec_gas * BASE_P["price_gas"]          # GBP/kg (energy only)
print(f"Gas SEC {sec_gas:.3f} kWh/kg ({sec_gas*3.6:.2f} MJ/kg), oven-only boundary, vs sources: "
      f"Carbon Trust 0.221 / Paton 0.239 / Spooner ~0.25 / Beech 0.29-0.32 kWh/kg: OK")
print(f"Gas baking emissions {per_kg_emis*1000:.0f} g CO2e/kg; oven energy cost {per_kg_cost*100:.2f} p/kg "
      f"(oven-only; excludes proving, cooling and other plant loads)")

# Embodied-emissions screening (3.2): literature-proxy equipment embodied carbon
# vs 15 years of energy-related emissions - defends excluding embodied emissions.
EQUIP_MASS_T, EMBODIED_T_PER_T = 40.0, 3.0            # provisional proxies
embodied = EQUIP_MASS_T * EMBODIED_T_PER_T            # tCO2e
lifetime_gas = lifetime_emissions(BASE_P, GAS)
print(f"Embodied-emissions screening: ~{embodied:.0f} t vs {lifetime_gas:,.0f} t lifetime gas energy emissions "
      f"({100*embodied/lifetime_gas:.1f}%) -> exclusion defensible")
results["verification"] = {
    "gas_per_kg_emissions_kgCO2e": per_kg_emis,
    "gas_per_kg_energy_cost_GBP": per_kg_cost,
    "embodied_screening_t": embodied,
    "embodied_share_of_lifetime_%": 100 * embodied / lifetime_gas,
}



## 7&nbsp;&nbsp;Illustration of the system boundary
Gate-to-gate baking step plus upstream energy carriers.

In [ ]:
# F1: system boundary (gate-to-gate baking step + upstream energy carriers)

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.axis("off")
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)


def draw_box(x, y, w, h, text, facecolour):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                fc=facecolour, ec="#333", lw=1.3, alpha=0.9))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)


def draw_arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>",
                                 mutation_scale=14, lw=1.4, color="#555"))


ax.text(5, 6.7, "System boundary: baking step (gate-to-gate) + upstream energy carriers",
        ha="center", fontsize=12, weight="bold")
draw_box(0.3, 4.6, 2.4, 1.1, "Energy supply\n(natural gas / grid electricity)", "#e9e2d0")
draw_box(0.3, 2.0, 2.4, 1.1, "Upstream (WTT) fuel\n& electricity emissions",    "#d7e3ee")
draw_box(3.7, 3.3, 2.6, 1.7, "OVEN\nGas / Electric / Hybrid\n(efficiency measures)", "#f2ead9")
draw_box(7.3, 4.6, 2.4, 1.1, "Baked product\n(15-30 kt/yr)",                    "#dcefe2")
draw_box(7.3, 2.0, 2.4, 1.1, "Direct (Scope 1) +\nupstream CO2e",              "#f0d9d6")
draw_box(3.7, 0.5, 2.6, 1.0, "TEA: CAPEX, OPEX,\nNPC, MAC (" + GBP + ")",       "#e7e7e7")
draw_arrow(2.7, 5.15, 3.7, 4.4)
draw_arrow(2.7, 2.55, 3.7, 3.7)
draw_arrow(6.3, 4.4, 7.3, 5.15)
draw_arrow(6.3, 3.7, 7.3, 2.55)
draw_arrow(5.0, 3.3, 5.0, 1.5)
save_figure(fig, "f_method_system_boundary.png")




## 8&nbsp;&nbsp;Objective 1: Baseline energy, cost and direct emissions
*Determine the energy demand, cost and direct emissions of a representative gas-fired baking oven as a baseline, including the effect of efficiency-first measures.*

In [ ]:
# OBJECTIVE 1 - BASELINE: ENERGY, COST AND DIRECT EMISSIONS

energy_band      = {c: scalar_band(lambda P, c=c: annual_energy(P, c) / 1e6) for c in CONFIGS}   # GWh
emis_lc_band     = {c: scalar_band(lambda P, c=c: annual_emissions_lc(P, c)) for c in CONFIGS}   # t (2024 grid)
emis_direct_band = {c: scalar_band(lambda P, c=c: annual_emissions_direct(P, c)) for c in CONFIGS}
emis_life_band   = {c: scalar_band(lambda P, c=c: lifetime_emissions(P, c) / 1e3) for c in CONFIGS}  # kt

for c in CONFIGS:
    results["energy"][c] = {
        "SEC_low_base_high": sec_display(c),
        "annual_GWh_base": energy_band[c][0],
        "annual_GWh_p5": energy_band[c][1], "annual_GWh_p95": energy_band[c][3],
    }
    results["emissions"][c] = {
        "direct_tCO2_yr_base": emis_direct_band[c][0],
        "opup_tCO2e_yr_2024grid_base": emis_lc_band[c][0],
        "opup_tCO2e_yr_2024grid_p5": emis_lc_band[c][1],
        "opup_tCO2e_yr_2024grid_p95": emis_lc_band[c][3],
        "lifetime_ktCO2e_base_traj": emis_life_band[c][0],
        "lifetime_ktCO2e_p5": emis_life_band[c][1], "lifetime_ktCO2e_p95": emis_life_band[c][3],
    }

#F2: annual energy demand ----------------
fig, ax = plt.subplots(figsize=(8, 4.6))
bar_colours = [COLOUR["gas"], COLOUR["gas_eff"], COLOUR["elec"], COLOUR["hybrid"]]
energy_base = [energy_band[c][0] for c in CONFIGS]
ax.bar(range(4), energy_base, color=bar_colours, width=0.6,
       yerr=err_bars([energy_band[c] for c in CONFIGS]), capsize=5, ecolor="#333")
for i, c in enumerate(CONFIGS):
    lo, _, hi = sec_display(c)
    ax.text(i, energy_band[c][3] + 0.4, f"{energy_base[i]:.1f} GWh\nSEC {lo:.2f}-{hi:.2f}",
            ha="center", fontsize=8)
ax.set_xticks(range(4))
ax.set_xticklabels(wrap_labels(CONFIGS), fontsize=9)
ax.set_ylabel("Annual final energy (GWh/yr)")
ax.set_ylim(0, max(b[3] for b in energy_band.values()) + 6)
ax.set_title("Annual energy demand (bars=base, error bars=P5-P95)")
save_figure(fig, "f_obj1_1_energy.png")

write_csv("f_obj1_1_energy.csv",
          ["configuration", "SEC_low", "SEC_base", "SEC_high",
           "annual_energy_GWh_base", "annual_energy_GWh_p5", "annual_energy_GWh_p50", "annual_energy_GWh_p95"],
          [[c, *sec_display(c), *energy_band[c]] for c in CONFIGS])




## 9&nbsp;&nbsp;Objective 2: Operating Expenses (OPEX) and Net Present Cost of Ownership (NPC) across spark gaps
*Quantify the capital and operating costs of electrified and hybrid oven configurations under a range of electricity-to-gas price ratios and tariff structures.*

In [ ]:
# OBJECTIVE 2 - COSTS: OPEX AND NPC ACROSS SPARK GAPS

opex_band = {c: scalar_band(lambda P, c=c: annual_opex(P, c) / 1e6) for c in CONFIGS}
npc_band  = {c: scalar_band(lambda P, c=c: npc_total(P, c) / 1e6) for c in CONFIGS}

for c in CONFIGS:
    results["cost_base"][c] = {"opex_M_base": opex_band[c][0],
                               "opex_M_p5": opex_band[c][1], "opex_M_p95": opex_band[c][3]}
    results["npc"][c] = {"npc_M_base": npc_band[c][0],
                         "npc_M_p5": npc_band[c][1], "npc_M_p95": npc_band[c][3]}

# Annual operating cost, stacked components 

fig, ax = plt.subplots(figsize=(8, 4.6))
comp = {c: [v / 1e6 for v in opex_components(BASE_P, c)] for c in CONFIGS}
stack_labels = ["Energy Consumed", "Capacity Charge", "CCL",
                "Scope 1 Carbon", "Fixed O&M"]
stack_colours = ["#5b8fb0", "#7f6fa8", "#c98a2b", COLOUR["carbon"], "#9a9a9a"]
bottom = np.zeros(4)
for k, (lab, col) in enumerate(zip(stack_labels, stack_colours)):
    vals = np.array([comp[c][k] for c in CONFIGS])
    ax.bar(range(4), vals, bottom=bottom, color=col, label=lab)
    bottom += vals
ax.errorbar(range(4), bottom, yerr=err_bars([opex_band[c] for c in CONFIGS]),
            fmt="none", ecolor="#222", capsize=5)
ax.set_xticks(range(4))
ax.set_xticklabels(wrap_labels(CONFIGS), fontsize=9)
ax.set_ylabel(f"Annual operating cost (million {GBP}/yr)")
ax.set_title("Annual operating cost (stacked=base, whiskers=P5-P95)")
ax.legend(fontsize=9)
save_figure(fig, "f_obj2_1_opex.png")

write_csv("f_obj2_1_opex.csv",
          ["configuration", "energy_M", "capacity_M", "ccl_M", "carbon_scope1_M", "om_M",
           "total_opex_M_base", "total_opex_M_p5", "total_opex_M_p50", "total_opex_M_p95"],
          [[c, *comp[c], *opex_band[c]] for c in CONFIGS])

# NPC versus spark gap (electricity price varies, gas held at base)

fig, ax = plt.subplots(figsize=(8, 4.6))
spark_range = np.linspace(1, 5, 21)
npc_curves = {}
for config, colour in zip(CONFIGS, bar_colours):
    eval_fn = lambda P, s, config=config: npc_total(override(P, spark_gap=s), config) / 1e6
    base, p5, p95 = curve_band(spark_range, eval_fn)
    npc_curves[config] = (base, p5, p95)
    ax.plot(spark_range, base, color=colour, lw=2.0, label=config)
    ax.fill_between(spark_range, p5, p95, color=colour, alpha=0.15)
ax.axvline(RANGE["spark_gap"][1], color=COLOUR["carbon"], ls=":", lw=1.6)
ax.set_xlabel("Spark gap (P_elec / P_gas)")
ax.set_ylabel(f"15-yr NPC (million {GBP})")
ax.set_title("Net present cost vs spark gap (lines=base, shaded=P5-P95)")
ax.legend(fontsize=8)
save_figure(fig, "f_obj2_2_npc.png")

header = ["spark_gap"] + [f"{c} {s}" for c in CONFIGS for s in ("base", "p5", "p95")]
write_csv("f_obj2_2_npc.csv", header,
          [[spark_range[i]] + [npc_curves[c][k][i] for c in CONFIGS for k in range(3)]
           for i in range(len(spark_range))])



## 10&nbsp;&nbsp;Objective 3(a): Emissions under present and projected grids
*Quantify the life-cycle greenhouse-gas emissions of each configuration under present and projected grid carbon-intensity scenarios.*

In [ ]:
# OBJECTIVE 3 - EMISSIONS UNDER PRESENT AND PROJECTED GRIDS

crossover_band = scalar_band(crossover_grid_ef)
mean_traj_ef = float(BASE_GRID_TRAJ.mean())
results["crossover_grid_EF"] = {
    "base": crossover_band[0], "p5": crossover_band[1], "p95": crossover_band[3],
    "lifetime_mean_grid_EF_base_traj": mean_traj_ef,
    "note": "electric is emissions-better over the life when the lifetime-average "
            "grid EF is below EF*; single-year and trajectory bases both reported (3.6)",
}

# per-kg emissions vs grid intensity (static basis, EF* marked)
fig, ax = plt.subplots(figsize=(8, 4.6))
grid_range = np.linspace(0, 0.4, 41)
gas_base, gas_p5, _, gas_p95 = scalar_band(lambda P: P["sec"][GAS] * P["ef_gas_lc"] * 1000.0)
gaseff_base = BASE_P["sec"][GAS_EFF] * BASE_P["ef_gas_lc"] * 1000.0
ax.axhline(gas_base, color=COLOUR["gas"], lw=2.2, label=f"Gas baseline ({gas_base:.0f} g/kg)")
ax.axhspan(gas_p5, gas_p95, color=COLOUR["gas"], alpha=0.12)
ax.axhline(gaseff_base, color=COLOUR["gas_eff"], lw=1.8, ls="--",
           label=f"Gas + efficiency ({gaseff_base:.0f} g/kg)")
elec_base, elec_p5, elec_p95 = curve_band(grid_range, lambda P, g: P["sec"][ELEC] * g * 1000.0)
hyb_base, hyb_p5, hyb_p95 = curve_band(grid_range, lambda P, g: P["sec"][HYBRID] * g * 1000.0)
ax.plot(grid_range, elec_base, color=COLOUR["elec"], lw=2.2, label="Electric (ER)")
ax.fill_between(grid_range, elec_p5, elec_p95, color=COLOUR["elec"], alpha=0.15)
ax.plot(grid_range, hyb_base, color=COLOUR["hybrid"], lw=2.0, ls="--", label="Hybrid (ER+IR)")
ax.fill_between(grid_range, hyb_p5, hyb_p95, color=COLOUR["hybrid"], alpha=0.12)
ax.axvline(crossover_band[0], color="#777", ls=":", lw=1.5)
ax.text(crossover_band[0] + 0.005, 20, f"EF* = {crossover_band[0]:.2f} kg/kWh", fontsize=9)
short_names = {
    "UK 2024 base (DEFRA)": "2024 (DEFRA)",
    "UK operational 2023 (NESO)*": "2023 (op.)",
    "UK 2030 (NESO)": "2030",
    "UK 2035 (NESO, floored at 0)": "2035 (floored)",
    "High-carbon (counterfactual)": "counterfactual",
}
for i, (name, factor) in enumerate(GRID_SCENARIOS.items()):
    y = BASE_P["sec"][ELEC] * factor * 1000.0
    ax.scatter([factor], [y], color="#333", zorder=5, s=26)
    dy = 13 if i % 2 == 0 else -17
    ax.annotate(short_names.get(name, name), (factor, y), fontsize=9,
                xytext=(7, dy), textcoords="offset points", ha="left",
                arrowprops=dict(arrowstyle="-", lw=0.5, color="#999"))
ax.set_xlabel("Grid carbon intensity (kg CO2e/kWh)")
ax.set_ylabel("Baking emissions (g CO2e/kg)")
ax.set_title("Operational + upstream emissions vs grid intensity (shaded=P5-P95)")
ax.legend(fontsize=9)
save_figure(fig, "f_obj3_1_emissions.png")

write_csv("f_obj3_1_emissions.csv",
          ["grid_EF_kg_per_kWh", "gas_g_per_kg_base", "gas_g_per_kg_p5", "gas_g_per_kg_p95",
           "gas_eff_g_per_kg_base", "electric_g_per_kg_base", "electric_g_per_kg_p5",
           "electric_g_per_kg_p95", "hybrid_g_per_kg_base", "hybrid_g_per_kg_p5", "hybrid_g_per_kg_p95"],
          [[grid_range[i], gas_base, gas_p5, gas_p95, gaseff_base,
            elec_base[i], elec_p5[i], elec_p95[i], hyb_base[i], hyb_p5[i], hyb_p95[i]]
           for i in range(len(grid_range))])

# lifetime emissions along grid trajectories (dynamic basis)

fig, ax = plt.subplots(figsize=(8, 4.6))
life_base = [lifetime_emissions(BASE_P, c) / 1e3 for c in CONFIGS]                     # NESO base traj
life_slow = [lifetime_emissions(BASE_P, c, traj=SLOW_GRID_TRAJ) / 1e3 for c in CONFIGS]
x = np.arange(4)
ax.bar(x - 0.19, life_base, width=0.36, color=bar_colours, label="Projected trajectory (15-yr input)",
       yerr=err_bars([emis_life_band[c] for c in CONFIGS]), capsize=4, ecolor="#333")
ax.bar(x + 0.19, life_slow, width=0.36, color=bar_colours, alpha=0.45,
       hatch="//", label="High-carbon counterfactual (0.134 flat)")
ax.set_xticks(x)
ax.set_xticklabels(wrap_labels(CONFIGS), fontsize=9)
ax.set_ylabel("Lifetime emissions, 2025-2039 (ktCO2e)")
ax.set_title("Lifetime operational + upstream emissions by grid trajectory")
ax.legend(fontsize=9)
save_figure(fig, "f_obj3_2_lifetime_emissions.png")

write_csv("f_obj3_2_lifetime_emissions.csv",
          ["configuration", "lifetime_kt_projection_base", "lifetime_kt_p5", "lifetime_kt_p95",
           "lifetime_kt_high_carbon"],
          [[c, life_base[i], emis_life_band[c][1], emis_life_band[c][3], life_slow[i]]
           for i, c in enumerate(CONFIGS)])




## 11&nbsp;&nbsp;Objective 3(b) Grid-intensity scenario ensemble (four FES scenarions and full Monte Carlo)
Lifetime emissions by configuration across the four NESO pathways, each with its full Monte-Carlo band.

Each of the four projection scenarios is carried through the FULL Monte Carlo. Scenario axis = between-pathway grid uncertainty; within-scenario MC = all OTHER. Pparameters include (SEC, production, CAPEX, prices, discount rate...)

In [ ]:


results["grid_scenarios"] = {}
scen_life, scen_mac = {}, {}
for _sname, _T in GRID_SCENARIOS_TRAJ.items():
    _life = {c: scalar_band(lambda P, c=c, T=_T: lifetime_emissions(P, c, traj=T) / 1e3)
             for c in CONFIGS}
    _mac  = scalar_band(lambda P, T=_T: mac_lifetime(P, traj=T))
    scen_life[_sname], scen_mac[_sname] = _life, _mac
    results["grid_scenarios"][_sname] = {
        "mean_grid_EF": float(_T.mean()),
        "lifetime_ktCO2e": {c: {"base": _life[c][0], "p5": _life[c][1], "p95": _life[c][3]}
                            for c in CONFIGS},
        "mac_lifetime_GBP_per_t": {"base": _mac[0], "p5": _mac[1], "p95": _mac[3]},
        "electric_beats_gas_on_life": bool(_life[ELEC][0] < _life[GAS][0]),
    }

# --- F5c: lifetime emissions by configuration across the four scenarios (each with MC band)
fig, ax = plt.subplots(figsize=(9.2, 4.8))
_snames = list(GRID_SCENARIOS_TRAJ)
_xg = np.arange(len(CONFIGS))
_w = 0.8 / len(_snames)
for _j, _sname in enumerate(_snames):
    _vals = [scen_life[_sname][c][0] for c in CONFIGS]
    _errs = err_bars([scen_life[_sname][c] for c in CONFIGS])
    ax.bar(_xg + (_j - (len(_snames) - 1) / 2) * _w, _vals, width=_w * 0.95,
           yerr=_errs, capsize=3, ecolor="#333", label=_sname)
ax.set_xticks(_xg); ax.set_xticklabels(wrap_labels(CONFIGS), fontsize=9)
ax.set_ylabel("Lifetime emissions, 2025-2039 (ktCO2e)")
ax.set_title("Lifetime emissions across four grid scenarios (bars=base, whiskers=P5-P95)")
ax.legend(fontsize=8, ncol=2)
save_figure(fig, "f_obj3b_1_scenario_ensemble.png")

write_csv("f_obj3b_1_scenario_ensemble.csv",
          ["scenario", "mean_grid_EF"] +
          sum([[f"{c}_base_kt", f"{c}_p5", f"{c}_p95"] for c in CONFIGS], []) +
          ["mac_life_base", "mac_life_p5", "mac_life_p95"],
          [[_s, GRID_SCENARIOS_TRAJ[_s].mean()] +
           sum([[scen_life[_s][c][0], scen_life[_s][c][1], scen_life[_s][c][3]] for c in CONFIGS], []) +
           [scen_mac[_s][0], scen_mac[_s][1], scen_mac[_s][3]]
           for _s in _snames])


## 12&nbsp;&nbsp;Objective 4: Demand-side flexibility
*Determine the additional economic and emissions impact of demand-side flexibility on oven operation* (average vs marginal emission-factor basis).

In [ ]:

results["flexibility"] = {}
for config in [ELEC, HYBRID]:
    bands = {
        "cost_firm_M": scalar_band(lambda P, c=config: firm_cost(P, c) / 1e6),
        "cost_flex_M": scalar_band(lambda P, c=config: flex_cost(P, c) / 1e6),
        "emis_firm_avg_t": scalar_band(lambda P, c=config: firm_emis(P, c, "average")),
        "emis_flex_avg_t": scalar_band(lambda P, c=config: flex_emis(P, c, "average")),
        "emis_firm_marg_t": scalar_band(lambda P, c=config: firm_emis(P, c, "marginal")),
        "emis_flex_marg_t": scalar_band(lambda P, c=config: flex_emis(P, c, "marginal")),
    }
    results["flexibility"][config] = {k: v[0] for k, v in bands.items()}
    if config == ELEC:
        flex_bands = bands

fb = flex_bands
fig, axs = plt.subplots(1, 3, figsize=(12, 4.3))
axs[0].bar(["Firm", "Flexible"], [fb["cost_firm_M"][0], fb["cost_flex_M"][0]],
           color=[COLOUR["elec"], COLOUR["hybrid"]],
           yerr=err_bars([fb["cost_firm_M"], fb["cost_flex_M"]]), capsize=5, ecolor="#333")
axs[0].set_title("Annual electricity cost")
axs[0].set_ylabel(f"million {GBP}/yr")
axs[1].bar(["Firm", "Flexible"], [fb["emis_firm_avg_t"][0], fb["emis_flex_avg_t"][0]],
           color=[COLOUR["elec"], COLOUR["hybrid"]],
           yerr=err_bars([fb["emis_firm_avg_t"], fb["emis_flex_avg_t"]]), capsize=5, ecolor="#333")
axs[1].set_title("Emissions - AVERAGE EF basis")
axs[1].set_ylabel("tCO2e/yr")
axs[2].bar(["Firm", "Flexible"], [fb["emis_firm_marg_t"][0], fb["emis_flex_marg_t"][0]],
           color=[COLOUR["elec"], COLOUR["hybrid"]],
           yerr=err_bars([fb["emis_firm_marg_t"], fb["emis_flex_marg_t"]]), capsize=5, ecolor="#333")
axs[2].set_title("Emissions - MARGINAL EF basis")
axs[2].set_ylabel("tCO2e/yr")
fig.suptitle("Demand-side flexibility, electric oven (error bars=P5-P95); "
             "k_p/k_e provisional pending half-hourly NESO data", fontsize=11)
save_figure(fig, "f_obj4_1_flexibility.png")

write_csv("f_obj4_1_flexibility.csv",
          ["metric", "firm_base", "firm_p5", "firm_p95", "flex_base", "flex_p5", "flex_p95"],
          [["electricity_cost_M_GBP_yr", fb["cost_firm_M"][0], fb["cost_firm_M"][1], fb["cost_firm_M"][3],
            fb["cost_flex_M"][0], fb["cost_flex_M"][1], fb["cost_flex_M"][3]],
           ["emissions_avg_basis_t_yr", fb["emis_firm_avg_t"][0], fb["emis_firm_avg_t"][1], fb["emis_firm_avg_t"][3],
            fb["emis_flex_avg_t"][0], fb["emis_flex_avg_t"][1], fb["emis_flex_avg_t"][3]],
           ["emissions_marginal_basis_t_yr", fb["emis_firm_marg_t"][0], fb["emis_firm_marg_t"][1],
            fb["emis_firm_marg_t"][3], fb["emis_flex_marg_t"][0], fb["emis_flex_marg_t"][1],
            fb["emis_flex_marg_t"][3]]])

## 13&nbsp;&nbsp;Objective 5: Thresholds conditions break-even, MAC, LCOH and decision map
*Determine the conditions (price ratio, carbon price, grid carbon intensity) at which electrification becomes the least-cost or least-carbon option.*

dfdddfd

In [ ]:

carbon_prices = np.linspace(0, 136, 35)   # DESNZ (2026) traded carbon values

be_base, be_p5, be_p95 = curve_band(carbon_prices, lambda P, cp: breakeven_spark_gap(P, cp))
be_eff_base, be_eff_p5, be_eff_p95 = curve_band(
    carbon_prices, lambda P, cp: breakeven_spark_gap(P, cp, gas_config=GAS_EFF))

results["breakeven_spark_gap"] = {
    "at_C0_base": float(be_base[0]),
    "at_C41.84_base": float(np.interp(41.84, carbon_prices, be_base)),
    "at_C136_base": float(be_base[-1]),
}

mac_static_band = scalar_band(lambda P: mac_value(P, P["ef_grid_2024"]))
mac_life_band   = scalar_band(mac_lifetime)
becp_band       = scalar_band(breakeven_carbon_price)
diff_band       = scalar_band(lambda P: npc_diff_elec_gas(P))
base_npc_diff   = diff_band[0]

results["MAC_GBP_per_t"] = {
    "static_2024_grid": {"base": mac_static_band[0], "p5": mac_static_band[1], "p95": mac_static_band[3]},
    "lifetime_trajectory": {"base": mac_life_band[0], "p5": mac_life_band[1], "p95": mac_life_band[3]},
}
results["breakeven_carbon_price_GBP_per_t"] = {
    "base": becp_band[0], "p5": becp_band[1], "p95": becp_band[3],
}
results["npc_diff_elec_gas_M"] = {"base": diff_band[0], "p5": diff_band[1], "p95": diff_band[3]}

results["LCOH"] = {}
for c in CONFIGS:
    kg_c  = scalar_band(lambda P, c=c: lcoh(P, c, "kg", True))
    kwh_c = scalar_band(lambda P, c=c: lcoh(P, c, "kWh", True))
    results["LCOH"][c] = {
        "per_kg_with_carbon_base": kg_c[0], "per_kg_p5": kg_c[1], "per_kg_p95": kg_c[3],
        "per_kWh_final_with_carbon_base": kwh_c[0],
        "per_kg_without_carbon_base": lcoh(BASE_P, c, "kg", False),
        "denominator_note": "per kg product (primary) and per kWh final energy; "
                            "useful-heat denominator pending heat-utilisation data",
    }

# break-even spark gap vs carbon price

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(carbon_prices, be_base, color=COLOUR["elec"], lw=2.2, label="vs gas baseline")
ax.fill_between(carbon_prices, be_p5, be_p95, color=COLOUR["elec"], alpha=0.15)
ax.plot(carbon_prices, be_eff_base, color=COLOUR["gas_eff"], lw=2.2, ls="--", label="vs gas + efficiency")
ax.fill_between(carbon_prices, be_eff_p5, be_eff_p95, color=COLOUR["gas_eff"], alpha=0.12)
spark_mid = RANGE["spark_gap"][1]
ax.axhline(spark_mid, color=COLOUR["carbon"], ls=":", lw=1.6)
ax.axhspan(RANGE["spark_gap"][0], RANGE["spark_gap"][2], color=COLOUR["carbon"], alpha=0.07)
ax.text(70, spark_mid + 0.1, f"UK spark gap {spark_mid:.2f}", color=COLOUR["carbon"],
        fontsize=9, ha="center")
ax.set_xlabel(f"Carbon price ({GBP}/tCO2e, Scope 1 policy scenario)")
ax.set_ylabel("Break-even spark gap (P_elec / P_gas)")
ax.set_title("Price ratio at which electric matches gas (shaded=P5-P95)")
ax.legend(fontsize=9)
save_figure(fig, "f_obj5_1_breakeven.png")

write_csv("f_obj5_1_breakeven.csv",
          ["carbon_price_GBP_per_t", "vs_gas_base", "vs_gas_p5", "vs_gas_p95",
           "vs_gas_eff_base", "vs_gas_eff_p5", "vs_gas_eff_p95"],
          [[carbon_prices[i], be_base[i], be_p5[i], be_p95[i],
            be_eff_base[i], be_eff_p5[i], be_eff_p95[i]] for i in range(len(carbon_prices))])

# Decision map - spark gap x grid intensity
# Here the delta NPC (depends on the spark gap; carbon is Scope 1 only, so the grid axis affects EMISSIONS not cost).  Vertical isolines: NPC break-even
# spark gap at representative carbon prices.  Horizontal line: EF* (least-carbon threshold).  The lower-left region is both least-cost and least-carbon.

fig, ax = plt.subplots(figsize=(8, 5))
spark_axis = np.linspace(1, 5, 81)
grid_axis  = np.linspace(0.0, 0.4, 2)
diff_line = np.array([npc_diff_elec_gas(override(BASE_P, spark_gap=s)) for s in spark_axis])
Z = np.tile(diff_line, (2, 1))
max_abs = np.nanmax(np.abs(Z))
filled = ax.contourf(spark_axis, grid_axis, Z, levels=20, cmap="RdBu_r",
                     vmin=-max_abs, vmax=max_abs)
cb = fig.colorbar(filled)
cb.set_label(f"NPC(electric) - NPC(gas) [million {GBP}]\n<0 electric cheaper")

iso_prices = [0.0, RANGE["carbon_price"][1], 136.0]
for cp, ls in zip(iso_prices, ["--", "-", ":"]):
    s_star = breakeven_spark_gap_npc(BASE_P, carbon_price=cp)
    if 1 <= s_star <= 5:
        ax.axvline(s_star, color="k", ls=ls, lw=1.8)
        ax.text(s_star + 0.03, 0.365, f"c={cp:.0f}", fontsize=8, rotation=90)

# MC band on the base-carbon break-even spark gap

s_draws = np.array([breakeven_spark_gap_npc(P) for P in MC_CURVE])
s5, s95 = np.nanpercentile(s_draws, [5, 95])
ax.axvspan(max(1, s5), min(5, s95), color="k", alpha=0.06)
# Least-carbon threshold EF* (2024-grid basis) with MC band
ax.axhline(crossover_band[0], color=COLOUR["hybrid"], lw=2.0)
ax.axhspan(crossover_band[1], crossover_band[3], color=COLOUR["hybrid"], alpha=0.12)
ax.text(4.05, crossover_band[0] + 0.006, "EF* (least-carbon threshold)",
        color=COLOUR["hybrid"], fontsize=8)
ax.axvline(spark_mid, color=COLOUR["carbon"], ls="-.", lw=1.2)
ax.text(spark_mid - 0.45, 0.375, f"UK {spark_mid:.2f}", fontsize=8, color=COLOUR["carbon"])
ax.set_xlabel("Spark gap (P_elec / P_gas)")
ax.set_ylabel("Grid carbon intensity (kg CO2e/kWh)")
ax.set_title("Decision map: NPC break-even isolines by carbon price;\n"
             "below EF* and left of an isoline, electric is cheaper AND cleaner")
save_figure(fig, "f_obj5_2_decision_map.png")

results["decision_map"] = {
    "breakeven_spark_npc": {f"c={cp:.0f}": float(breakeven_spark_gap_npc(BASE_P, cp))
                            for cp in iso_prices},
    "breakeven_spark_npc_base_carbon_p5_p95": [float(s5), float(s95)],
}
write_csv("f_obj5_2_decision_map.csv",
          ["spark_gap", "npc_diff_M_base"] + [f"breakeven_spark_at_c{cp:.0f}" for cp in iso_prices],
          [[spark_axis[i], diff_line[i]] +
           [results["decision_map"]["breakeven_spark_npc"][f"c={cp:.0f}"] for cp in iso_prices]
           for i in range(len(spark_axis))])

#tornado - one-at-a-time low/high
def base_with(**top):
    return override(BASE_P, **top)

def base_with_sec(config, value):
    Q = override(BASE_P)
    Q["sec"] = dict(BASE_P["sec"])
    Q["sec"][config] = value
    return Q

def base_with_capex(config, value):
    Q = override(BASE_P)
    Q["capex"] = dict(BASE_P["capex"])
    Q["capex"][config] = value
    return Q

def lo_hi(key):
    return base_with(**{key: RANGE[key][0]}), base_with(**{key: RANGE[key][2]})

tornado_specs = [
    ("Spark gap",             *lo_hi("spark_gap")),
    ("Gas price",             *lo_hi("price_gas")),
    ("Carbon price",          *lo_hi("carbon_price")),
    ("Discount rate",         *lo_hi("discount_rate")),
    ("O&M fraction",          *lo_hi("om_fraction")),
    ("Production",            *lo_hi("production")),
    ("Grid connection CAPEX", *lo_hi("grid_connection")),
    ("Residual value",        *lo_hi("residual_fraction")),
    # CCL is statutory (not sampled); shown as a 0x-2x POLICY scenario (A)
    ("CCL abolished vs doubled (policy)",
     base_with(ccl_gas=0.0, ccl_elec=0.0),
     base_with(ccl_gas=2 * CCL_RATES["gas"], ccl_elec=2 * CCL_RATES["elec"])),
    ("Grid intensity (no NPC effect: carbon is Scope 1 only)", *lo_hi("ef_grid_2024")),
    ("Electric CAPEX",        base_with_capex(ELEC, CAPEX_RANGE[ELEC][0]),
                              base_with_capex(ELEC, CAPEX_RANGE[ELEC][2])),

    # Electric SEC saving d varied at the base gas SEC (low d = high electric SEC)

    ("Electric SEC saving d", base_with_sec(ELEC, SEC_RANGE[GAS][1] * (1 - RANGE["elec_saving"][0])),
                              base_with_sec(ELEC, SEC_RANGE[GAS][1] * (1 - RANGE["elec_saving"][2]))),
]

tornado = [(label, npc_diff_elec_gas(P_lo), npc_diff_elec_gas(P_hi))
           for label, P_lo, P_hi in tornado_specs]
tornado.sort(key=lambda row: abs(row[2] - row[1]))

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for i, (label, low, high) in enumerate(tornado):
    ax.barh(i, high - base_npc_diff, left=base_npc_diff, color="#b0483b", alpha=0.8)
    ax.barh(i, low - base_npc_diff, left=base_npc_diff, color="#2f6f9f", alpha=0.8)
ax.axvline(base_npc_diff, color="k", lw=1.5)
ax.text(base_npc_diff, len(tornado) - 0.3, f"base {base_npc_diff:+.1f}M", fontsize=9, ha="center")
ax.set_yticks(range(len(tornado)))
ax.set_yticklabels([row[0] for row in tornado], fontsize=8)
ax.set_xlabel(f"NPC(electric) - NPC(gas) [million {GBP}] (<0 favours electric)")
ax.set_title("One-at-a-time sensitivity (low/high, others at base)")
save_figure(fig, "f_obj5_3_tornado.png")

write_csv("f_obj5_3_tornado.csv",
          ["parameter", "npc_diff_low_M", "npc_diff_high_M", "base_npc_diff_M"],
          [[label, low, high, base_npc_diff] for (label, low, high) in tornado])

# Policy scenarios for the CCL treatment

_P_noccl = base_with(ccl_gas=0.0, ccl_elec=0.0)
_P_cca   = base_with(ccl_gas=CCL_RATES["gas"] * CCA_SHARE_PAID["gas"],
                     ccl_elec=CCL_RATES["elec"] * CCA_SHARE_PAID["elec"])
results["policy_scenarios_CCL"] = {
    "note": ("delta NPC (electric - gas, GBP m); CCL charged on both carriers at the "
             "stated rates.  CCA shares paid are indicative - verify HMRC reduced "
             "rates before citing."),
    "main_rates_npc_diff_M": float(npc_diff_elec_gas(BASE_P)),
    "no_CCL_npc_diff_M": float(npc_diff_elec_gas(_P_noccl)),
    "CCA_reduced_rates_npc_diff_M": float(npc_diff_elec_gas(_P_cca)),
    "CCA_share_paid": CCA_SHARE_PAID,
    "breakeven_spark_gap_at_base_carbon": {
        "main_rates": float(breakeven_spark_gap(BASE_P, BASE_P["carbon_price"])),
        "no_CCL": float(breakeven_spark_gap(_P_noccl, BASE_P["carbon_price"])),
        "CCA_reduced": float(breakeven_spark_gap(_P_cca, BASE_P["carbon_price"])),
    },
}

# Differentiated O&M scenario: gas burners vs electric elements

Q = override(BASE_P)
diff_om = (npc_total(Q, ELEC) - Q["capex"][ELEC] * (0.03 - 0.025) * annuity_factor(Q)) \
          - (npc_total(Q, GAS) - Q["capex"][GAS] * (0.03 - 0.04) * annuity_factor(Q))
results["differentiated_OM_scenario_M"] = {
    "assumption": "gas O&M 4%/yr, electric O&M 2.5%/yr (vs flat 3%)",
    "npc_diff_M": diff_om / 1e6,
}

### &nbsp;&nbsp;Marginal abatement cost (MAC) vs grid intensity

In [ ]:
# F10: MAC vs grid intensity
fig, ax = plt.subplots(figsize=(8, 4.6))
grid_for_mac = np.linspace(0.0, 0.23, 47)
mac_base, mac_p5, mac_p95 = curve_band(grid_for_mac, lambda P, g: mac_value(P, g))
ax.plot(grid_for_mac, mac_base, color=COLOUR["elec"], lw=2.2, label="Static grid EF (eq 3.14)")
ax.fill_between(grid_for_mac, mac_p5, mac_p95, color=COLOUR["elec"], alpha=0.15)
ax.axhline(mac_life_band[0], color=COLOUR["hybrid"], ls="--", lw=1.8,
           label=f"Lifetime-trajectory MAC ({mac_life_band[0]:.0f}/t)")
ax.axhline(RANGE["carbon_price"][1], color=COLOUR["carbon"], ls="--", lw=1.2)
ax.text(0.15, RANGE["carbon_price"][1] * 1.4, f"UK ETS {GBP}{RANGE['carbon_price'][1]:.0f}/t",
        color=COLOUR["carbon"], fontsize=9)
ax.set_xlabel("Grid carbon intensity (kg CO2e/kWh)")
ax.set_ylabel(f"Marginal abatement cost ({GBP}/tCO2e)")
ax.set_title("Cost of abating CO2e by electrifying (shaded=P5-P95)")
ax.legend(fontsize=9)
save_figure(fig, "f_obj5_4_mac.png")

write_csv("f_obj5_4_mac.csv",
          ["grid_EF_kg_per_kWh", "mac_static_base", "mac_static_p5", "mac_static_p95",
           "mac_lifetime_trajectory_base"],
          [[grid_for_mac[i], mac_base[i], mac_p5[i], mac_p95[i], mac_life_band[0]]
           for i in range(len(grid_for_mac))])


### &nbsp;&nbsp;Global sensitivity standardised regression coefficients (SRC)

In [ ]:
# --- F11: global sensitivity - standardised regression coefficients
SRC_KEYS = [
    ("spark_gap", "Spark gap"), ("price_gas", "Gas price"), ("carbon_price", "Carbon price"),
    ("discount_rate", "Discount rate"), ("om_fraction", "O&M fraction"),
    ("production", "Production"), ("eff_saving", "Efficiency saving e"),
    ("capacity_charge", "Capacity charge q"), ("grid_connection", "Grid connection CAPEX"),
    ("residual_fraction", "Residual value"),
    ("ef_grid_2024", "Grid intensity"), ("sec_gas_raw", "Gas SEC"),
    ("elec_saving", "Electric SEC saving d"),
    ("carbon_growth", "Carbon price growth"), ("gas_growth", "Gas price growth"),
    ("elec_growth", "Electricity price growth"),
]
y = np.array([npc_diff_elec_gas(P) for P in MC_PARAMS])
X = np.column_stack([[P[k] for P in MC_PARAMS] for k, _ in SRC_KEYS]
                    + [[P["capex"][GAS] for P in MC_PARAMS], [P["capex"][ELEC] for P in MC_PARAMS]])
src_labels = [lab for _, lab in SRC_KEYS] + ["Gas CAPEX", "Electric CAPEX"]
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
ys = (y - y.mean()) / y.std()
beta, *_ = np.linalg.lstsq(np.column_stack([Xs, np.ones(len(ys))]), ys, rcond=None)
src = beta[:-1]
r2 = 1 - np.sum((ys - np.column_stack([Xs, np.ones(len(ys))]) @ beta) ** 2) / np.sum(ys ** 2)
order = np.argsort(np.abs(src))
results["SRC_global_sensitivity"] = {
    "R2": float(r2), "note": "SRC on delta NPC; extend to Sobol if R2 low / interactions material",
    "coefficients": {src_labels[i]: float(src[i]) for i in np.argsort(-np.abs(src))},
}

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.barh(range(len(src)), src[order],
        color=["#2f6f9f" if v < 0 else "#b0483b" for v in src[order]])
ax.set_yticks(range(len(src)))
ax.set_yticklabels([src_labels[i] for i in order], fontsize=8)
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("Standardised regression coefficient on delta NPC (electric - gas)")
ax.set_title(f"Global sensitivity ranking from {N_MC:,} MC draws (R2={r2:.2f})")
save_figure(fig, "f_obj5_5_src.png")

write_csv("f_obj5_5_src.csv", ["parameter", "SRC"],
          [[src_labels[i], src[i]] for i in np.argsort(-np.abs(src))])

#Monte Carlo convergence check ----------------------------------------
half = np.nanpercentile(y[: N_MC // 2], [5, 50, 95])
full = np.nanpercentile(y, [5, 50, 95])
conv = np.max(np.abs(full - half) / np.abs(full))
results["mc_convergence"] = {
    "delta_npc_P5_P50_P95_at_half_N": half.tolist(),
    "delta_npc_P5_P50_P95_at_full_N": full.tolist(),
    "max_relative_shift": float(conv),
    "converged_within_1pct": bool(conv < 0.01),
}

## 14&nbsp;&nbsp;Layered cost analysis and break-even CAPEX
NPC decomposed into capital and operating layers; the electric-oven capital cost at which NPC(electric)=NPC(gas).

In [ ]:
MILLION = 1e6

# Layer 1 = operating cost only (capex removed).  Layer 2 = full NPC (capex + opex).
def pv_operating_cost(P, config, carbon_price=None):
    """Discounted lifetime operating cost in GBP (energy, capacity, CCL, carbon, O&M).
       It is the full NPC with the initial capital and the residual value removed."""
    full_npc     = npc_total(P, config, carbon_price)
    capital_out  = capex_initial(P, config)
    residual_val = P["residual_fraction"] * P["capex"][config] * discount_factors(P)[-1]
    return full_npc - capital_out + residual_val

# Break-even electric-oven capex: the electric capex that makes NPC(electric) = NPC(gas).
# NPC is a straight line in capex, so we evaluate it at two capex values and read off zero.
def breakeven_electric_capex(P, spark_gap=None, carbon_price=None):
    Q = P
    if spark_gap is not None:
        Q = override(P, spark_gap=spark_gap)          # also updates the electricity price
    def npc_gap(capex_value):                          # NPC(electric) - NPC(gas)
        R = dict(Q)
        R["capex"] = dict(Q["capex"])
        R["capex"][ELEC] = capex_value
        return npc_total(R, ELEC, carbon_price) - npc_total(R, GAS, carbon_price)
    capex_a, capex_b = 1.0 * MILLION, 2.0 * MILLION
    gap_a = npc_gap(capex_a)
    gap_b = npc_gap(capex_b)
    slope = (gap_b - gap_a) / (capex_b - capex_a)
    return capex_a - gap_a / slope                     # capex where the gap is zero

# --- Layer table (base case): annual opex, lifetime opex, capex, full NPC ---
layer_rows = []
for c in CONFIGS:
    layer_rows.append([
        c,
        annual_opex(BASE_P, c) / MILLION,
        pv_operating_cost(BASE_P, c) / MILLION,
        capex_initial(BASE_P, c) / MILLION,
        npc_total(BASE_P, c) / MILLION,
    ])
write_csv("f_obj5_6_cost_layers.csv",
          ["configuration", "annual_opex_Mgbp_yr", "lifetime_opex_PV_Mgbp",
           "initial_capex_Mgbp", "full_npc_Mgbp"],
          layer_rows)

# NPC split into capital and operating cost (stacked bars)

capex_vals = [capex_initial(BASE_P, c) / MILLION for c in CONFIGS]
opex_vals  = [pv_operating_cost(BASE_P, c) / MILLION for c in CONFIGS]
x = np.arange(len(CONFIGS))
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.bar(x, capex_vals, color="#4477aa", label="Capital (initial outlay)")
ax.bar(x, opex_vals, bottom=capex_vals, color="#ee6677", label="Operating cost (PV over life)")
ax.set_xticks(x); ax.set_xticklabels(wrap_labels(CONFIGS), fontsize=9)
ax.set_ylabel("Net present cost (GBP million)")
ax.set_title("Where the cost sits: capital vs operating (base case)")
ax.legend()
save_figure(fig, "f_obj5_6_cost_layers.png")

# Break-even electric capex at the actual spark gap

be_capex_now = breakeven_electric_capex(BASE_P)
results["breakeven_electric_capex"] = {
    "at_actual_spark_gap_Mgbp": be_capex_now / MILLION,
    "assumed_electric_capex_Mgbp": BASE_P["capex"][ELEC] / MILLION,
    "note": "electric-oven capex that ties gas on NPC; negative => no oven price makes electric least-cost",
}

# break-even electric capex vs spark gap, at three carbon prices

spark_values = np.linspace(0.5, 5.0, 46)
carbon_cases = [0.0, 41.84, 136.0]
curve_rows = []
for sg in spark_values:
    row = [sg]
    for cp in carbon_cases:
        row.append(breakeven_electric_capex(BASE_P, spark_gap=sg, carbon_price=cp) / MILLION)
    curve_rows.append(row)
write_csv("f_obj5_7_breakeven_capex_vs_sparkgap.csv",
          ["spark_gap", "breakeven_capex_Mgbp_carbon0",
           "breakeven_capex_Mgbp_carbon42", "breakeven_capex_Mgbp_carbon136"],
          curve_rows)

fig, ax = plt.subplots(figsize=(8, 4.8))
for i, cp in enumerate(carbon_cases):
    ax.plot(spark_values, [r[i + 1] for r in curve_rows], label=f"carbon = £{cp:.0f}/t")
ax.axhline(0, color="#333", lw=1)
ax.axhline(BASE_P["capex"][ELEC] / MILLION, color="#999", ls=":",
           label=f"assumed electric capex (£{BASE_P['capex'][ELEC]/MILLION:.1f}M)")
ax.axvline(BASE_P["spark_gap"], color="#cc3311", ls="--",
           label=f"actual spark gap ({BASE_P['spark_gap']:.2f})")
ax.set_xlabel("Spark gap (electricity price / gas price)")
ax.set_ylabel("Break-even electric-oven capex (GBP million)")
ax.set_title("How cheap the electric oven must be to match gas on NPC")
ax.legend(fontsize=8)
save_figure(fig, "f_obj5_7_breakeven_capex_vs_sparkgap.png")

# break-even capex surface over spark gap x carbon price

sg_axis = np.linspace(0.5, 5.0, 19)         # X axis (columns)
cp_axis = np.linspace(0.0, 300.0, 16)       # Y axis (rows)
surface = np.zeros((len(cp_axis), len(sg_axis)))
for i in range(len(cp_axis)):
    for j in range(len(sg_axis)):
        surface[i, j] = breakeven_electric_capex(
            BASE_P, spark_gap=sg_axis[j], carbon_price=cp_axis[i]) / MILLION

fig, ax = plt.subplots(figsize=(8, 5))
shaded = ax.contourf(sg_axis, cp_axis, surface, levels=20, cmap="RdYlGn")
line = ax.contour(sg_axis, cp_axis, surface,
                  levels=[BASE_P["capex"][ELEC] / MILLION], colors="black", linewidths=2)
ax.clabel(line, fmt="assumed capex")
fig.colorbar(shaded, label="Break-even electric capex (GBP million)")
ax.axvline(BASE_P["spark_gap"], color="#0033aa", ls="--",
           label=f"actual spark gap ({BASE_P['spark_gap']:.2f})")
ax.set_xlabel("Spark gap (electricity price / gas price)")
ax.set_ylabel("Carbon price (GBP / tCO2e)")
ax.set_title("Break-even electric-oven capex across price and carbon conditions")
ax.legend(fontsize=8, loc="upper right")
save_figure(fig, "f_obj5_8_breakeven_capex_surface.png")

# MATRIX csv: first column = carbon price, first row = spark gaps.

matrix_header = ["carbon_price_GBP_per_t"] + [f"sg_{sg:.2f}" for sg in sg_axis]
matrix_rows = []
for i in range(len(cp_axis)):
    matrix_rows.append([f"{cp_axis[i]:.0f}"] + [f"{surface[i, j]:.3f}" for j in range(len(sg_axis))])
write_csv("f_obj5_8_breakeven_capex_surface_matrix.csv", matrix_header, matrix_rows)


## 15&nbsp;&nbsp;Key results summary
Consolidated printout of the headline figures.

In [ ]:
print("\n================ KEY RESULTS (layered) ================")
print(f"Operating-cost parity spark gap (capex-free): {results['breakeven_spark_gap']['at_C0_base']:.2f} "
      f"(actual {BASE_P['spark_gap']:.2f})")
print(f"Full-NPC gap electric - gas: £{results['npc_diff_elec_gas_M']['base']:.1f}M")
print(f"Break-even electric-oven capex at actual prices: £{be_capex_now/MILLION:.1f}M "
      f"(assumed £{BASE_P['capex'][ELEC]/MILLION:.1f}M) -> "
      f"{'capex NOT binding' if be_capex_now < 0 else 'capex binding'}")
print(f"Break-even carbon price: £{results['breakeven_carbon_price_GBP_per_t']['base']:.0f}/t")
print(f"Lifetime MAC (base trajectory): £{results['MAC_GBP_per_t']['lifetime_trajectory']['base']:.0f}/t")
print("=======================================================\n")


with open(os.path.join(HERE, "results_v5.json"), "w") as f:
    json.dump(results, f, indent=2, default=float)

print()
print("=== KEY RESULTS (real 2024 GBP) - base [P5, P95] ===")
print(f"Monte Carlo draws: {N_MC:,}  |  convergence (half vs full N): "
      f"max shift {100*conv:.2f}% -> {'OK' if conv < 0.01 else 'CHECK'}")
print(f"Base spark gap (ex-CCL): {spark_mid:.2f}; CCL-inclusive price ratio: "
      f"{results['base_price_ratio_incl_CCL']:.2f}")
print(f"Policy costs (2.7): CCL {CCL_RATES['gas']*100:.3f} p/kWh on BOTH carriers; "
      f"carbon {RANGE['carbon_price'][1]}/t on Scope 1 (on-site gas) ONLY - "
      f"not applied to electricity (embedded in tariff)")
ps = results["policy_scenarios_CCL"]
print(f"CCL policy scenarios, delta NPC electric-gas (M): main {ps['main_rates_npc_diff_M']:+.2f} / "
      f"no-CCL {ps['no_CCL_npc_diff_M']:+.2f} / CCA {ps['CCA_reduced_rates_npc_diff_M']:+.2f}; "
      f"SG* at base carbon: main {ps['breakeven_spark_gap_at_base_carbon']['main_rates']:.3f} / "
      f"no-CCL {ps['breakeven_spark_gap_at_base_carbon']['no_CCL']:.3f}")
for c in CONFIGS:
    print(f"  {c}: energy {energy_band[c][0]:.1f} [{energy_band[c][1]:.1f}, {energy_band[c][3]:.1f}] GWh; "
          f"lifetime emis {emis_life_band[c][0]:.1f} [{emis_life_band[c][1]:.1f}, {emis_life_band[c][3]:.1f}] kt; "
          f"NPC {npc_band[c][0]:.1f} [{npc_band[c][1]:.1f}, {npc_band[c][3]:.1f}] {GBP} M")
print(f"Emissions-parity grid EF* (2024 basis): {crossover_band[0]:.3f} "
      f"[{crossover_band[1]:.3f}, {crossover_band[3]:.3f}] kg/kWh "
      f"(lifetime-average base-trajectory EF = {mean_traj_ef:.3f} -> electric emissions-better over life)")
print(f"NPC diff electric-gas: {diff_band[0]:+.1f} [{diff_band[1]:+.1f}, {diff_band[3]:+.1f}] {GBP} M")
print(f"MAC static 2024 grid: {mac_static_band[0]:.0f} [{mac_static_band[1]:.0f}, {mac_static_band[3]:.0f}] /t; "
      f"lifetime-trajectory: {mac_life_band[0]:.0f} [{mac_life_band[1]:.0f}, {mac_life_band[3]:.0f}] /t")
print(f"Break-even carbon price: {becp_band[0]:.0f} [{becp_band[1]:.0f}, {becp_band[3]:.0f}] {GBP}/t")
print(f"NPC break-even spark gap at c=0/{RANGE['carbon_price'][1]:.0f}/136: "
      + " / ".join(f"{breakeven_spark_gap_npc(BASE_P, cp):.2f}" for cp in iso_prices))
print("LCOH GBP/kg with carbon (base):",
      {k.split(" (")[0]: round(v["per_kg_with_carbon_base"], 4) for k, v in results["LCOH"].items()})
print("Figures:", sorted(os.listdir(FIG_DIR)))
print("CSV datasets:", sorted(os.listdir(CSV_DIR)))


## 16&nbsp;&nbsp;Time to parity
Year at which the gas oven's annual operating cost catches the electric oven, under real price and carbon-price growth, with Monte-Carlo uncertainty.

To find year at which the gas oven's annual operating cost captches up with the electric oven's, as the real carbon price rises.
Since electric oven is operating-cost-heavy, parity is driven by carbon growth. The plot shows years-to-parity vs the assumed real carbon-growth rate, with a P5-P95 band across all other Monte Carlo uncertainty.

In [ ]:


PARITY_HORIZON = 60   # search out to 60 yr so a parity year can be reported

def annual_opex_year(P, config, k):
    """Annual operating cost in year k (0-based), with real price/carbon growth."""
    E = annual_energy(P, config)
    if is_gas(config):
        energy   = E * P["price_gas"] * (1 + P["gas_growth"]) ** k
        capacity = 0.0
        carbon   = annual_emissions_direct(P, config) * P["carbon_price"] * (1 + P["carbon_growth"]) ** k
    else:
        p_energy = max(P["price_elec"] - P["capacity_charge"] / HOURS, 0.0)
        energy   = E * p_energy * (1 + P["elec_growth"]) ** k
        capacity = P["capacity_charge"] * peak_demand(P, config)   # network charge, not escalated
        carbon   = 0.0
    levy = E * ccl_rate(P, config)
    om   = P["om_fraction"] * P["capex"][config]
    return energy + capacity + levy + carbon + om

def years_to_parity(P):
    """First year (1-based) gas annual cost >= electric annual cost; NaN if none in horizon."""
    for k in range(PARITY_HORIZON):
        if annual_opex_year(P, GAS, k) >= annual_opex_year(P, ELEC, k):
            return k + 1
    return np.nan

# Sweep the real carbon-growth rate; band = P5-P95 across the MC draws

growth_axis = np.linspace(0.0, 0.25, 26)
ttp_base, ttp_p5, ttp_med, ttp_p95, frac_in_life = [], [], [], [], []
for g in growth_axis:
    base_val = years_to_parity(override(BASE_P, carbon_growth=g))
    draws = np.array([years_to_parity(override(P, carbon_growth=g)) for P in MC_CURVE])
    filled = np.where(np.isnan(draws), PARITY_HORIZON, draws)        # censor non-parity at horizon
    p5, med, p95 = np.percentile(filled, [5, 50, 95])
    ttp_base.append(PARITY_HORIZON if np.isnan(base_val) else base_val)
    ttp_p5.append(p5); ttp_med.append(med); ttp_p95.append(p95)
    frac_in_life.append(float(np.mean(np.nan_to_num(draws, nan=1e9) <= LIFE)))

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.fill_between(growth_axis * 100, ttp_p5, ttp_p95, color=COLOUR["elec"], alpha=0.15, label="P5-P95")
ax.plot(growth_axis * 100, ttp_med, color=COLOUR["elec"], lw=2.2, label="Median")
ax.plot(growth_axis * 100, ttp_base, color=COLOUR["carbon"], lw=1.6, ls="--", label="Base case")
ax.axhline(LIFE, color="#333", lw=1.4, ls=":")
ax.text(0.3, LIFE + 0.8, f"asset life = {LIFE} yr", fontsize=9, color="#333")
ax.set_ylim(0, PARITY_HORIZON)
ax.set_xlabel("Assumed real carbon-price growth rate (%/yr)")
ax.set_ylabel("Years to operating-cost parity (gas catches electric)")
ax.set_title("Time to parity vs carbon-price growth (band = P5-P95 Monte Carlo)")
ax.legend(fontsize=9, loc="upper right")
save_figure(fig, "f_obj5_9_time_to_parity.png")

write_csv("f_obj5_9_time_to_parity.csv",
          ["carbon_growth_pct_per_yr", "years_to_parity_base", "years_to_parity_p5",
           "years_to_parity_median", "years_to_parity_p95", "fraction_of_draws_parity_within_life"],
          [[round(growth_axis[i] * 100, 2), ttp_base[i], ttp_p5[i], ttp_med[i], ttp_p95[i],
            round(frac_in_life[i], 3)] for i in range(len(growth_axis))])

# find the growth rate at which the MEDIAN reaches parity within the asset life

_med = np.array(ttp_med)
_within = growth_axis[_med <= LIFE]
print("\n[Time to parity] median years-to-parity at base carbon growth "
      f"({RANGE['carbon_growth'][1]*100:.0f}%/yr): "
      f"{ttp_med[int(np.argmin(np.abs(growth_axis - RANGE['carbon_growth'][1])))]:.0f} yr")
print("[Time to parity] real carbon growth needed for median parity WITHIN the "
      f"{LIFE}-yr life: {_within.min()*100:.0f}%/yr" if _within.size else
      f"[Time to parity] median never reaches parity within {LIFE} yr even at 25%/yr")